In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Load Dataset

In [10]:
import random
from torch.utils.data import Subset

data_dir = "./imagenet_validation"      # Download from https://www.kaggle.com/datasets/tusonggao/imagenet-validation-dataset
synset2label = {}
with open(data_dir + "/labels.txt", "r") as f:
    for line in f:
        synset, label = line.strip().split(" ", 1)
        synset2label[synset] = label

transform = transforms.Compose([
    transforms.Resize(256),
    # transforms.CenterCrop(224),     # Default
    transforms.CenterCrop(256),   # SigLIP Large
    transforms.ToTensor(),  # just brings it to [0, 1] float tensor
])
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

clip_labels = []
for synset, idx in sorted(dataset.class_to_idx.items(), key=lambda x: x[1]):
    label = synset2label[synset]
    clip_labels.append(f"a photo of a {label}")

siglip_labels = []
for synset, idx in sorted(dataset.class_to_idx.items(), key=lambda x: x[1]):
    label = synset2label[synset]
    siglip_labels.append(f"This is a photo of a {label}")

n_samples = 5000
indices = random.sample(range(len(dataset)), n_samples)
subset = Subset(dataset, indices)

loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
)

num_classes = len(dataset.classes)
print("Dataset size:", len(loader.dataset))
print("Classes:", num_classes)

Dataset size: 5000
Classes: 1000


# Evaluate Models

In [4]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoProcessor


class ModelWrapper(nn.Module):
    def __init__(self, model_name, num_classes, device):
        super().__init__()
        self.name = model_name
        self.device = device

        # ===== CLIP =====
        if model_name == "clip_base":
            self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
            self.processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-base-patch16"
            )
            self.labels = clip_labels
        elif model_name == "clip_large":
            self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
            self.processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-large-patch14"
            )
            self.labels = clip_labels

        # ===== SigLIP =====
        elif model_name == "siglip_base":
            self.model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
            self.processor = AutoProcessor.from_pretrained(
                "google/siglip-base-patch16-224"
            )
            self.labels = siglip_labels
        elif model_name == "siglip_large":
            self.model = AutoModel.from_pretrained("google/siglip-large-patch16-256")
            self.processor = AutoProcessor.from_pretrained(
                "google/siglip-large-patch16-256"
            )
            self.labels = siglip_labels

        else:
            raise ValueError(f"Unknown model: {model_name}")

        self.model = self.model.to(device)
        self.eval()

        with torch.no_grad():
            text_inputs = self.processor(
                text=self.labels, return_tensors="pt", padding=True
            ).to(device)

            self.text_features = self.model.get_text_features(
                **text_inputs
            ).pooler_output
            self.text_features = self.text_features / self.text_features.norm(
                dim=-1, keepdim=True
            )

    def forward(self, images):
        with torch.no_grad():
            image_inputs = self.processor(
                images=images, return_tensors="pt", do_rescale=False
            ).to(self.device)
            image_features = self.model.get_image_features(**image_inputs).pooler_output
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            logits = image_features @ self.text_features.T
            return logits

In [5]:
from pathlib import Path
import torchvision.transforms.functional as TF
import time


def apply_affine_batch(images, angle, translate, scale, shear):
    return torch.stack(
        [
            TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear)
            for img in images
        ]
    )


def evaluate(
    model,
    loader,
    model_name: str = "Model",
    translation: tuple[float, float] = [0, 0],
    rotation: float = 0.0,
    scale: float = 1.0,
    shear: tuple[float, float] = [0, 0],
    save_results_path: str = "results.csv",
):
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = apply_affine_batch(
                images, angle=rotation, translate=translation, scale=scale, shear=shear
            )

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds_top1 = outputs.argmax(dim=1)
            correct_top1 += (preds_top1 == labels).sum().item()

            _, preds_top5 = outputs.topk(5, dim=1)
            correct_top5 += (preds_top5 == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    acc_top1 = correct_top1 / total
    acc_top5 = correct_top5 / total

    end_time = time.time()
    eval_time = end_time - start_time

    results_path = Path(save_results_path)
    header = (
        "Model Name,Dataset Size,Translation X, Translation Y,"
        "Rotation,Scale,Shear X, Shear Y,Top-1 Accuracy,Top-5 Accuracy,Time"
    )

    if not results_path.exists() or results_path.stat().st_size == 0:
        # file doesn't exist or is empty -> create with header
        with results_path.open("w", encoding="utf-8") as f:
            f.write(header + "\n")
    else:
        # file exists, check header consistency
        with results_path.open("r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        if first_line != header:
            with results_path.open("w", encoding="utf-8") as f:
                f.write(header + "\n")

    # append result row
    with results_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{model_name},{len(loader.dataset)},"
            f"{translation[0]},{translation[1]},{rotation},{scale},"
            f"{shear[0]},{shear[1]},{acc_top1:.4f},{acc_top5:.4f},{eval_time:.2f}\n"
        )

    return acc_top1, acc_top5

In [6]:
def evaluate_architecture(model_name, loader, geometric_transforms):
    model = ModelWrapper(model_name, num_classes, device)

    # Baseline evaluation without transformations
    if geometric_transforms["baseline"]:
        print(f"Baseline (No Transformations):")
        acc_top1, acc_top5 = evaluate(model, loader, model_name)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Translations
    for translation in geometric_transforms["translations"]:
        print(f"Translation {translation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, translation=translation)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Rotations
    for rotation in geometric_transforms["rotations"]:
        print(f"Rotation {rotation}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, rotation=rotation)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Scales
    for scale in geometric_transforms["scales"]:
        print(f"Scale {scale}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, scale=scale)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

    # Shears
    for shear in geometric_transforms["shears"]:
        print(f"Shear {shear}:")
        acc_top1, acc_top5 = evaluate(model, loader, model_name, shear=shear)
        print(f"Top-1 Accuracy: {acc_top1 * 100:.2f}%")
        print(f"Top-5 Accuracy: {acc_top5 * 100:.2f}%")

### CLIP ViT Base

In [7]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("clip_base", loader, geometric_transforms)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.29it/s]


Top-1 Accuracy: 64.20%
Top-5 Accuracy: 89.98%
Translation (25, 0):


...

Top-1 Accuracy: 32.90%
Top-5 Accuracy: 58.80%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:19<00:00,  4.13it/s]


Top-1 Accuracy: 31.64%
Top-5 Accuracy: 56.78%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:20<00:00,  3.81it/s]

Top-1 Accuracy: 32.26%
Top-5 Accuracy: 57.58%


### CLIP ViT Large

In [8]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("clip_large", loader, geometric_transforms)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline (No Transformations):


 90%|██████████████████████████████████████████████████████████████████████████▌        | 71/79 [00:56<00:06,  1.29it/s]

100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.28it/s]


...

Top-1 Accuracy: 47.08%
Top-5 Accuracy: 72.48%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.29it/s]


Top-1 Accuracy: 46.14%
Top-5 Accuracy: 71.04%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.27it/s]

Top-1 Accuracy: 45.60%
Top-5 Accuracy: 71.44%


### SigLIP Base

In [9]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("siglip_base", loader, geometric_transforms)

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:20<00:00,  3.95it/s]


Top-1 Accuracy: 47.96%
Top-5 Accuracy: 70.02%
Translation (25, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:19<00:00,  4.10it/s]


...

Top-1 Accuracy: 24.52%
Top-5 Accuracy: 44.98%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:19<00:00,  4.07it/s]


Top-1 Accuracy: 22.32%
Top-5 Accuracy: 41.10%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [00:20<00:00,  3.78it/s]

Top-1 Accuracy: 24.50%
Top-5 Accuracy: 44.12%


### SigLIP Large

In [11]:
geometric_transforms = {
    "baseline": True,
    "translations": [
        (25, 0),  # Translate 25 pixels along X-axis
        (0, 25),  # Translate 25 pixels along Y-axis
        (-25, 0),  # Translate -25 pixels along X-axis
        (0, -25),  # Translate -25 pixels along Y-axis
        (50, 0),  # Translate 50 pixels along X-axis
        (0, 50),  # Translate 50 pixels along Y-axis
        (-50, 0),  # Translate -50 pixels along X-axis
        (0, -50),  # Translate -50 pixels along Y-axis
    ],
    "rotations": [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330],
    "scales": [0.5, 1.5, 2.0],
    "shears": [
        (25, 0),  # Shear 25 degrees along X-axis
        (0, 25),  # Shear 25 degrees along Y-axis
        (-25, 0),  # Shear -25 degrees along X-axis
        (0, -25),  # Shear -25 degrees along Y-axis
        (50, 0),  # Shear 50 degrees along X-axis
        (0, 50),  # Shear 50 degrees along Y-axis
        (-50, 0),  # Shear -50 degrees along X-axis
        (0, -50),  # Shear -50 degrees along Y-axis
    ],
}

evaluate_architecture("siglip_large", loader, geometric_transforms)

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

Baseline (No Transformations):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:03<00:00,  1.25it/s]


Top-1 Accuracy: 25.10%
Top-5 Accuracy: 41.92%
Translation (25, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.28it/s]


...

Top-1 Accuracy: 15.46%
Top-5 Accuracy: 28.08%
Shear (-50, 0):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:01<00:00,  1.29it/s]


Top-1 Accuracy: 14.78%
Top-5 Accuracy: 26.54%
Shear (0, -50):


100%|███████████████████████████████████████████████████████████████████████████████████| 79/79 [01:03<00:00,  1.24it/s]


Top-1 Accuracy: 15.02%
Top-5 Accuracy: 27.34%
